<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.1-rag-fundamentals/notebooks/exercises-6_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.1: RAG Fundamentals in Python

*Module 6 · Retrieval-Augmented Generation (RAG)*

> LLMs don’t know YOUR data. RAG fixes this: retrieve relevant documents, inject into the prompt, generate grounded answers. This lesson builds a complete RAG pipeline from scratch — no LangChain, no LlamaIndex — so you understand every component.

`Module 6` · `Pure Python` · `No Frameworks` · `Full Pipeline` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-6.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Step 1: Chunk Documents — `01_chunking.py`
2. Step 2 — Steps 2-5: Complete RAG Pipeline in One Class — `02_complete_rag.py`
3. Step 3 — RAG With vs Without — See the Difference — `03_comparison.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Step 1: Chunk Documents

Split into retrievable pieces with overlap.

**`01_chunking.py`** · _Block 1 of 3_

RAG STEP 1: CHUNKING


In [ ]:
# RAG STEP 1: CHUNKING
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        if len(chunk.split()) >= 20:
            chunks.append({"id": len(chunks), "text": chunk})
    return chunks

doc = "Netsetos is an edtech company in Hyderabad..." * 20
chunks = chunk_text(doc, chunk_size=60, overlap=10)
print(f"Document: {len(doc.split())} words → {len(chunks)} chunks")
print(f"Sweet spot: 200-500 words, 50-word overlap")


> **What just happened?** Documents split into overlapping chunks. Overlap prevents losing info at boundaries. Chunk size is the #1 tuning parameter: too small = fragments. Too large = noise. Sweet spot: 200-500 words.


### Step 2 · Steps 2-5: Complete RAG Pipeline in One Class

Embed, store, retrieve, augment, generate — all in ~60 lines.

**`02_complete_rag.py`** · _Block 2 of 3_

COMPLETE RAG PIPELINE FROM SCRATCH — No frameworks


In [ ]:
# COMPLETE RAG PIPELINE FROM SCRATCH — No frameworks
import google.generativeai as genai
import numpy as np, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class SimpleRAG:
    def __init__(self, chunk_size=60, overlap=10, top_k=3):
        self.chunk_size, self.overlap, self.top_k = chunk_size, overlap, top_k
        self.chunks, self.embeddings = [], []
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def add_document(self, text, source="doc"):
        words = text.split()
        for i in range(0, len(words), self.chunk_size - self.overlap):
            chunk = " ".join(words[i:i+self.chunk_size])
            if len(chunk.split()) < 15: continue
            emb = genai.embed_content(model="models/text-embedding-004", content=chunk)["embedding"]
            self.chunks.append({"text": chunk, "source": source})
            self.embeddings.append(np.array(emb))
        print(f"  Indexed: {len(self.chunks)} chunks from '{source}'")

    def retrieve(self, query, k=None):
        k = k or self.top_k
        qe = np.array(genai.embed_content(model="models/text-embedding-004", content=query)["embedding"])
        scores = [(i, np.dot(qe,e)/(np.linalg.norm(qe)*np.linalg.norm(e))) for i,e in enumerate(self.embeddings)]
        scores.sort(key=lambda x:-x[1])
        return [{"text":self.chunks[i]["text"],"score":s,"source":self.chunks[i]["source"]} for i,s in scores[:k]]

    def query(self, question):
        docs = self.retrieve(question)
        context = "\n\n".join([f"[{d['source']}] {d['text']}" for d in docs])
        prompt = f"Answer ONLY from this context. If not found, say so.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
        resp = self.model.generate_content(prompt)
        return {"answer": resp.text.strip(), "sources": [{"text":d["text"][:60],"score":f"{d['score']:.3f}"} for d in docs]}

# ══ BUILD AND TEST ══
rag = SimpleRAG(chunk_size=60, overlap=10, top_k=3)
print("Building RAG...\n")
rag.add_document("Netsetos is an edtech company in Hyderabad offering GenAI, Data Science, Cloud courses. Flagship: GenAI Complete Course, 14 modules, 146 hours.", "about.txt")
rag.add_document("Refund policy: Full refund within 7 days. 50 percent from 8-30 days. No refunds after 30 days. Processed in 5 business days.", "refund.txt")
rag.add_document("GenAI course: 14999 rupees. Includes lifetime access, Discord, weekly live sessions, certificate, 5000 GCP credits. EMI via Razorpay.", "pricing.txt")
rag.add_document("Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. Python and GCP. 85 percent completion. 92 percent placement. 4.8 star rating. 5000 students.", "faq.txt")

print("\nRAG Q&A:\n")
for q in ["What is the refund policy?", "How much does GenAI cost?", "When are live classes?", "Does Netsetos offer blockchain?"]:
    r = rag.query(q)
    print(f"  Q: {q}\n  A: {r['answer'][:120]}\n")


> **What just happened?** Complete RAG in ~60 lines: 4 docs chunked, embedded, indexed. Questions matched by cosine similarity, top-3 chunks injected into prompt, Gemini answers from context. The blockchain question returns “I don’t have this information.” No hallucination. Grounded in YOUR data.


### Step 3 · RAG With vs Without — See the Difference

Same question. With context (accurate) vs without (hallucination).

**`03_comparison.py`** · _Block 3 of 3_

RAG vs NO RAG — Side by side


In [ ]:
# RAG vs NO RAG — Side by side
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

q = "What is Netsetos's refund policy?"
context = "Refund: Full within 7 days. 50% from 8-30 days. None after 30 days."

no_rag = model.generate_content(q)
with_rag = model.generate_content(f"Answer ONLY from context.\nContext: {context}\nQ: {q}")

print(f"WITHOUT RAG:\n  {no_rag.text.strip()[:150]}\n")
print(f"WITH RAG:\n  {with_rag.text.strip()[:150]}\n")
print("Without: hallucinated or 'I don't know'. With: exact policy from YOUR docs.")


> **What just happened?** Document processing is the most impactful and most overlooked stage. The recommended pipeline: PDF → pymupdf4llm → Markdown → RecursiveCharacterTextSplitter(512, overlap=50) . Extract metadata (source, page, section) for filtering. For tables, keep them intact as Markdown — don't chunk across rows. Parent-child retrieval resolves the fundamental tension between retrieval precision (small chunks) and generation context (large chunks).


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
